# Preprocessing – Smoking & Drinking Dataset

## Objective
Build a reproducible preprocessing pipeline to transform raw health data into a
clean, model-ready dataset for alcohol consumption prediction.

## Input
Raw dataset:
- `data/raw/smoking_driking_dataset_Ver01.csv`

## Output
Processed dataset and preprocessing artifacts to be used in downstream modeling.

## Data loading and initial setup

### Imports & configuración

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")

RANDOM_SEED = 42

### Define data paths

In [3]:
DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


### Load raw dataset

In [4]:
raw_path = RAW_DIR / "smoking_driking_dataset_Ver01.csv"

if not raw_path.exists():
    raise FileNotFoundError(
        f"Raw dataset not found at: {raw_path}\n"
        "Please download it from Kaggle and place it in data/raw/"
    )

df_raw = pd.read_csv(raw_path)
df = df_raw.copy()

df.shape

(991346, 24)

### Sanity check

In [5]:
df.head()

,sex,age,height,weight,waistline,sight_left,sight_right,hear_left,hear_right,SBP,...,LDL_chole,triglyceride,hemoglobin,urine_protein,serum_creatinine,SGOT_AST,SGOT_ALT,gamma_GTP,SMK_stat_type_cd,DRK_YN
0,Male,35,170,75,90.0,1.0,1.0,1.0,1.0,120.0,...,126.0,92.0,17.1,1.0,1.0,21.0,35.0,40.0,1.0,Y
1,Male,30,180,80,89.0,0.9,1.2,1.0,1.0,130.0,...,148.0,121.0,15.8,1.0,0.9,20.0,36.0,27.0,3.0,N
2,Male,40,165,75,91.0,1.2,1.5,1.0,1.0,120.0,...,74.0,104.0,15.8,1.0,0.9,47.0,32.0,68.0,1.0,N
3,Male,50,175,80,91.0,1.5,1.2,1.0,1.0,145.0,...,104.0,106.0,17.6,1.0,1.1,29.0,34.0,18.0,1.0,N
4,Male,50,165,60,80.0,1.0,1.2,1.0,1.0,138.0,...,117.0,104.0,13.8,1.0,0.8,19.0,12.0,25.0,1.0,N


## Target and feature definition

In this step, the target variable is defined and features are separated from labels.
No preprocessing is applied to features at this stage; encoding, imputation, and
scaling will be handled later using reproducible pipelines.

### Define target variable

In [6]:
TARGET_COL = "DRK_YN"

# Validate target column presence
if TARGET_COL not in df.columns:
    raise KeyError(f"Target column '{TARGET_COL}' not found in the dataset.")


### Separate features and labels

In [7]:
# Separate features and target (raw features, no preprocessing applied)
X = df.drop(columns=[TARGET_COL]).copy()
y_raw = df[TARGET_COL].copy()

X.shape, y_raw.shape

((991346, 23), (991346,))

### Map target to binary

In [8]:
# Map target variable from Y/N to binary labels

y = y_raw.map({"N": 0, "Y": 1})

# Sanity checks
if y.isna().any():
    invalid = y_raw[ y.isna() ].value_counts()
    raise ValueError(f"Unexpected values in target '{TARGET_COL}':\n{invalid}")

y.value_counts(), y.value_counts(normalize=True) * 100


(DRK_YN
 0    495858
 1    495488
 Name: count, dtype: int64,
 DRK_YN
 0    50.018661
 1    49.981339
 Name: proportion, dtype: float64)

### Identify feature types

In [9]:
# Identify categorical and numerical feature columns
categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()

categorical_cols, len(categorical_cols), len(numeric_cols)


(['sex'], 1, 22)

### Feature cardinality and encoding rationale

In [10]:
# Inspect cardinality of categorical and ordinal-like features
X[categorical_cols + ["SMK_stat_type_cd"]].nunique()

sex                 2
SMK_stat_type_cd    3
dtype: int64

### Feature cardinality and encoding rationale

- `sex` has low cardinality and represents a nominal category, making it suitable
  for one-hot encoding.
- `SMK_stat_type_cd` has three discrete values with an inherent order
  (never, former, current smoker), supporting its treatment as an ordinal numerical
  feature rather than a nominal categorical variable.